### Train Model

In [5]:
pip install scikit-learn


     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB 217.9 kB/s eta 0:00:41
     ---------------------------------------- 0.0/8.9 MB 187.9 kB/s eta 0:00:48
     ---------------------------------------- 0.0/8.9 MB 196.9 kB/s eta 0:00:45
     ---------------------------------------- 0.1/8.9 MB 252.2 kB/s eta 0:00:35
     ---------------------------------------- 0.1/8.9 MB 306.3 kB/s eta 0:00:29
     ---------------------------------------- 0.1/8.9 MB 306.3 kB/s eta 0:00:29
     ---------------------------------------- 0.1/8.9 MB 228.2 kB/s eta 0:00:39
      --------------------------------------- 0.1/8.9 MB 273.1 kB/s eta 0:00:33
      --------------------------------------- 0.2/8.9 MB 364.0 kB/s eta 0:00:24
      --------------------------------------- 0.2/8.9 MB 414.8 kB/


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pickle
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

# Load data
data = np.loadtxt("./data/data.txt")

X = data[:, :-1]
y = data[:, -1]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('knn', KNeighborsClassifier())
])

# Grid Search
params = {
    'pca__n_components': [20, 30, 50],
    'knn__n_neighbors': [3, 5, 7],
    'knn__weights': ['uniform', 'distance']
}

grid = GridSearchCV(
    pipeline,
    params,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

model = grid.best_estimator_

print("Best Params:", grid.best_params_)

# Evaluation
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy * 100:.2f}%")
print(confusion_matrix(y_test, y_pred))

# Save model
with open('./model.pkl', 'wb') as f:
    pickle.dump(model, f)

Best Params: {'pca__n_components': 30, 'svm__C': 20, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 47.83%
[[7 6 2]
 [4 6 6]
 [3 3 9]]


### Test Model

In [ ]:
import pickle
import cv2

from utils import get_face_landmarks

# نفس ترتيب التدريب
EMOTIONS = ['Happy', 'Sad', 'Surprised']

with open('./model', 'rb') as f:
    model = pickle.load(f)

cap = cv2.VideoCapture(0)  # جرب 0 لو 2 مش شغال

while True:
    ret, frame = cap.read()
    if not ret:
        break

    face_landmarks = get_face_landmarks(frame, draw=True, static_image_mode=False)

    if face_landmarks is not None and len(face_landmarks) == 1404:

        output = model.predict([face_landmarks])
        emotion = EMOTIONS[int(output[0])]

        cv2.putText(frame,
                    emotion,
                    (10, frame.shape[0] - 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    (0, 255, 0),
                    3)

    cv2.imshow('Emotion Detection', frame)

    # exit بـ ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

: 